# 02 — Model 

Churn risk model on KKBox. Inputs built in notebook 01, persisted in `data/churn.duckdb`:
- `pred_points` — raw labelled prediction points (per cycle)
- `cohorts`     — labelable points + temporal split (train / val / test)
- `cohorts_s`   — sampled spine for fast iteration (~300k / 100k / 100k)

Plan: commitment features + logistic baseline (PR-AUC floor) → add engagement / tenure / trend
→ XGBoost → calibrate → cost-based threshold.

In [ ]:
import duckdb, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)

con = duckdb.connect('../data/churn.duckdb')   
q = lambda sql: con.execute(sql).df()

DATA = "../data/"


In [9]:
q("""
CREATE OR REPLACE TABLE pred_points AS
WITH per_day AS (
    SELECT msno, strptime(transaction_date::VARCHAR,'%Y%m%d')::DATE AS txn_dt,
           max(strptime(membership_expire_date::VARCHAR,'%Y%m%d')::DATE) AS exp_dt
    FROM transactions GROUP BY msno, transaction_date),
pts AS (SELECT msno, exp_dt AS expiry, max(txn_dt) AS gov_txn FROM per_day GROUP BY msno, exp_dt),
labeled AS (
    SELECT p.msno, p.expiry, p.gov_txn,
      NOT EXISTS (SELECT 1 FROM per_day d WHERE d.msno=p.msno AND d.txn_dt>p.gov_txn
                  AND d.txn_dt<=p.expiry+30 AND d.exp_dt>p.expiry) AS is_churn,
      (p.expiry+30 <= DATE '2017-03-31' AND p.expiry >= DATE '2015-01-01') AS labelable
    FROM pts p)
SELECT msno, expiry, gov_txn, is_churn, date_trunc('month', expiry) AS cohort_month
FROM labeled WHERE labelable
""")
q("""
CREATE OR REPLACE TABLE cohorts AS
SELECT *, CASE WHEN cohort_month <= DATE '2016-08-01' THEN 'train'
               WHEN cohort_month <= DATE '2016-11-01' THEN 'val' ELSE 'test' END AS split
FROM pred_points WHERE cohort_month BETWEEN DATE '2015-02-01' AND DATE '2017-02-01'
""")
q("""
CREATE OR REPLACE TABLE cohorts_s AS
WITH ranked AS (SELECT *, row_number() OVER (PARTITION BY split ORDER BY hash(msno || '|' || expiry::VARCHAR)) AS rn
                FROM cohorts)
SELECT * EXCLUDE (rn) FROM ranked WHERE rn <= CASE split WHEN 'train' THEN 300000 ELSE 100000 END
""")
print(q("""SELECT split, count(*) AS n, round(avg(is_churn::int),4) AS churn_rate
           FROM cohorts_s GROUP BY split ORDER BY min(cohort_month)"""))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   split       n  churn_rate
0  train  300000      0.1106
1    val  100000      0.1202
2   test  100000      0.0628


In [10]:
q("""
CREATE OR REPLACE TABLE feat_commitment AS
SELECT c.msno, c.expiry, c.split, c.is_churn,
       t.is_auto_renew, t.payment_plan_days, t.actual_amount_paid, t.plan_list_price,
       (t.actual_amount_paid = 0)::int AS is_free,
       (t.plan_list_price - t.actual_amount_paid) AS discount,
       t.payment_method_id
FROM cohorts_s c
JOIN transactions t
  ON t.msno = c.msno
 AND t.transaction_date = strftime(c.gov_txn,'%Y%m%d')::INT
 AND t.membership_expire_date = strftime(c.expiry,'%Y%m%d')::INT
QUALIFY row_number() OVER (PARTITION BY c.msno, c.expiry ORDER BY t.actual_amount_paid DESC) = 1
""")
print(q("""SELECT is_auto_renew, count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
           FROM feat_commitment WHERE split='train' GROUP BY is_auto_renew ORDER BY is_auto_renew"""))
print(q("""SELECT is_free, count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
           FROM feat_commitment WHERE split='train' GROUP BY is_free ORDER BY is_free"""))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   is_auto_renew       n  churn_rate
0              0   43898       0.292
1              1  256102       0.080
   is_free       n  churn_rate
0        0  284242       0.072
1        1   15758       0.813


#### commitment features built (feat_commitment)
- join 1:1, 300k train, no row loss -> point-in-time join clean
- auto_renew: off 29% vs on 8% churn (3.6x)
- is_free: free 81% vs paid 7% churn (11x)
- both strong + cutoff-safe; is_cancel held back (leakage line)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
import pandas as pd

df = q("SELECT * FROM feat_commitment")
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','is_free','discount']
tr, va = df[df.split=='train'], df[df.split=='val']
ytr, yva = tr['is_churn'].astype(int), va['is_churn'].astype(int)
scaler = StandardScaler().fit(tr[feat_cols])
Xtr, Xva = scaler.transform(tr[feat_cols]), scaler.transform(va[feat_cols])
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
p_va = clf.predict_proba(Xva)[:,1]
print(f"val PR-AUC: {average_precision_score(yva, p_va):.4f}   (no-skill floor = {yva.mean():.4f})")
print(pd.Series(clf.coef_[0], index=feat_cols).sort_values())

val PR-AUC: 0.7595   (no-skill floor = 0.1202)
payment_plan_days    -1.212748
is_auto_renew        -0.422167
discount              0.198676
actual_amount_paid    0.594213
plan_list_price       0.703447
is_free               0.937151
dtype: float64


#### logistic baseline (commitment feats)
- val PR-AUC 0.76 vs no-skill 0.12 -> high for 6 feats -> interrogate, not celebrate
- NOT look-ahead leakage (all feats <= cutoff). driven by easy segments: free 81%, auto_renew-off 29%
  + val has 2016-11 promo spike that is_free nails
- coefs sane (plan_days -, is_free +, auto_renew -)
- open: signal among SAFE-looking (auto_renew=1 & paid)? = where the real value is [next]

In [12]:
mask = (va['is_auto_renew']==1) & (va['is_free']==0)
y_seg, p_seg = yva[mask], p_va[mask]
print(f"safe segment: n={mask.sum()}, base rate={y_seg.mean():.4f}, PR-AUC={average_precision_score(y_seg, p_seg):.4f}")
te = df[df.split=='test']; yte = te['is_churn'].astype(int)
p_te = clf.predict_proba(scaler.transform(te[feat_cols]))[:,1]
print(f"test PR-AUC={average_precision_score(yte, p_te):.4f}  (test base rate={yte.mean():.4f})")

safe segment: n=80918, base rate=0.0249, PR-AUC=0.0305
test PR-AUC=0.3127  (test base rate=0.0628)


#### baseline interrogated
- SAFE segment (auto_renew=1 & paid, 81% of val): base 2.5%, PR-AUC 0.031 = no-skill -> ZERO signal among safe-looking customers
  (partly mechanical: conditioned away the 2 strong feats - but that IS the point)
- TEST PR-AUC 0.31 (not 0.76): val flattered by higher base rate + 2016-11 promo spike. 0.31 = honest out-of-time floor (still >> 0.063)
- commitment feats READ obvious churns, can't find hidden risk -> need BEHAVIOURAL signal next

In [13]:
q("""
CREATE OR REPLACE TABLE feat_engagement AS
WITH pts AS (
    SELECT msno, expiry, strftime(expiry,'%Y%m%d')::INT AS e_int,
           strftime(expiry-30,'%Y%m%d')::INT AS e_30, strftime(expiry-60,'%Y%m%d')::INT AS e_60
    FROM cohorts_s),
j AS (
    SELECT p.msno, p.expiry, p.e_int, p.e_30, l.date, l.num_25, l.num_50, l.num_75, l.num_985,
           l.num_100, l.num_unq, l.total_secs
    FROM pts p JOIN user_logs l ON l.msno=p.msno AND l.date>p.e_60 AND l.date<=p.e_int),
agg AS (
    SELECT msno, expiry,
           date_diff('day', strptime(max(date)::VARCHAR,'%Y%m%d')::DATE, expiry) AS recency_days,
           count(DISTINCT CASE WHEN date>e_30 THEN date END) AS active_days_30,
           count(DISTINCT CASE WHEN date<=e_30 THEN date END) AS active_days_prior,
           sum(CASE WHEN date>e_30 THEN total_secs ELSE 0 END) AS secs_30,
           sum(CASE WHEN date>e_30 THEN num_unq ELSE 0 END) AS unq_30,
           sum(CASE WHEN date>e_30 THEN num_100 ELSE 0 END) AS completed_30,
           sum(CASE WHEN date>e_30 THEN num_25+num_50+num_75+num_985+num_100 ELSE 0 END) AS plays_30
    FROM j GROUP BY msno, expiry)
SELECT *, completed_30/nullif(plays_30,0) AS completion_ratio,
       active_days_30/nullif(active_days_prior,0) AS activity_trend
FROM agg
""")
print(q("SELECT count(*) AS n_rows, count(DISTINCT msno) AS users FROM feat_engagement"))
print(q("""
SELECT CASE WHEN recency_days<=3 THEN '0-3d' WHEN recency_days<=14 THEN '4-14d'
            WHEN recency_days<=30 THEN '15-30d' ELSE '31-60d' END AS recency_bucket,
       count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
FROM feat_engagement e JOIN cohorts_s c USING (msno, expiry)
WHERE c.split='train' GROUP BY recency_bucket ORDER BY churn_rate
"""))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_rows   users
0  401088  333549
  recency_bucket       n  churn_rate
0           0-3d  192767       0.071
1          4-14d   32552       0.145
2         31-60d    6362       0.155
3         15-30d   10875       0.200


#### engagement features (feat_engagement)
- 401k/500k points (80%) have logs in 60d window; ~20% SILENT -> dropped here, fill+flag at merge (likely highest churn)
- recency separates churn: 0-3d=7% vs 4-60d=14-20% (2-3x) -> real behavioural signal, varies within active users
- minor non-monotonic tail (15-30d 20% > 31-60d 15.5%) -> small-n + long-plan committed users quiet. not concerning

In [14]:
q("""
CREATE OR REPLACE TABLE model_data AS
SELECT c.*,
       (e.msno IS NOT NULL)::int AS has_activity_60d,
       coalesce(e.recency_days,60) AS recency_days,
       coalesce(e.active_days_30,0) AS active_days_30,
       coalesce(e.secs_30,0) AS secs_30,
       coalesce(e.unq_30,0) AS unq_30,
       coalesce(e.completion_ratio,0) AS completion_ratio,
       coalesce(e.activity_trend,0) AS activity_trend
FROM feat_commitment c
LEFT JOIN feat_engagement e USING (msno, expiry)
""")
print(q("SELECT count(*) AS n, count(*) FILTER (WHERE has_activity_60d=0) AS silent FROM model_data"))
print(q("""SELECT has_activity_60d, count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
           FROM model_data WHERE split='train' GROUP BY has_activity_60d ORDER BY has_activity_60d"""))

        n  silent
0  500000   98912
   has_activity_60d       n  churn_rate
0                 0   57444       0.202
1                 1  242556       0.089


#### merged model_data
- 500k restored (LEFT JOIN), 98,912 silent (no logs in 60d, ~20%)
- silent flag strong + cutoff-safe: silent 20.2% vs active 8.9% churn (2.3x)

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
import pandas as pd
df = q("SELECT * FROM model_data")
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','is_free','discount',
             'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
tr, va, te = df[df.split=='train'], df[df.split=='val'], df[df.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
scaler = StandardScaler().fit(tr[feat_cols])
Xtr, Xva, Xte = scaler.transform(tr[feat_cols]), scaler.transform(va[feat_cols]), scaler.transform(te[feat_cols])
clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
p_va, p_te = clf.predict_proba(Xva)[:,1], clf.predict_proba(Xte)[:,1]
print(f"val  PR-AUC: {average_precision_score(yva,p_va):.4f}   (commitment-only: 0.7595)")
print(f"test PR-AUC: {average_precision_score(yte,p_te):.4f}   (commitment-only: 0.3127)")
m = (va.is_auto_renew==1) & (va.is_free==0)
print(f"SAFE segment val: n={m.sum()}, base={yva[m].mean():.4f}, PR-AUC={average_precision_score(yva[m],p_va[m]):.4f}   (commitment-only: 0.0305)")
print(pd.Series(clf.coef_[0], index=feat_cols).sort_values())

val  PR-AUC: 0.8291   (commitment-only: 0.7595)
test PR-AUC: 0.4118   (commitment-only: 0.3127)
SAFE segment val: n=80918, base=0.0249, PR-AUC=0.0630   (commitment-only: 0.0305)
payment_plan_days    -1.674156
is_auto_renew        -0.572236
active_days_30       -0.379115
unq_30               -0.057782
secs_30               0.007217
completion_ratio      0.085525
activity_trend        0.140560
discount              0.286251
has_activity_60d      0.412879
recency_days          0.709540
actual_amount_paid    0.802000
is_free               0.851112
plan_list_price       0.958469
dtype: float64


#### combined logistic (commitment + engagement)
- val 0.76→0.83, test 0.31→0.41 (engagement lifts the honest out-of-time floor ~0.10)
- safe segment 0.03→0.063 (2.5x base): behaviour finds risk among look-safe customers; commitment was blind
- coef signs: payment_plan_days/auto_renew/active_days clean (−), recency/is_free clean (+)
- has_activity_60d flips +ve = collinear w/ recency (silent→recency=60 & activity=0); don't read logistic coefs as causal → motivates trees+SHAP

In [16]:
eng   = ['has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
price = ['payment_plan_days','actual_amount_paid','plan_list_price','discount','is_free']
print(tr[eng].corr().round(2)['recency_days'].sort_values())
print()
print(tr[price].corr().round(2))

has_activity_60d   -0.95
completion_ratio   -0.83
active_days_30     -0.72
unq_30             -0.42
activity_trend     -0.23
secs_30             0.02
recency_days        1.00
Name: recency_days, dtype: float64

                    payment_plan_days  actual_amount_paid  plan_list_price  \
payment_plan_days                1.00                0.85             0.96   
actual_amount_paid               0.85                1.00             0.88   
plan_list_price                  0.96                0.88             1.00   
discount                         0.20               -0.28             0.22   
is_free                         -0.08               -0.29            -0.11   

                    discount  is_free  
payment_plan_days       0.20    -0.08  
actual_amount_paid     -0.28    -0.29  
plan_list_price         0.22    -0.11  
discount                1.00     0.37  
is_free                 0.37     1.00  


#### collinearity confirmed (from data)
- recency_days ↔ has_activity_60d = −0.95 → one dormancy axis, explains the +ve sign flip
- payment_plan_days ↔ plan_list_price = 0.96, ↔ amount_paid 0.85 → one plan-size axis
- implication: don't read logistic coefs as drivers; trees + SHAP for attribution

In [17]:
from xgboost import XGBClassifier
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','is_free','discount',
             'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
df = q("SELECT * FROM model_data")
tr, va, te = df[df.split=='train'], df[df.split=='val'], df[df.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
clf = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, importance_type='gain', n_jobs=-1, random_state=42)
clf.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)
p_va = clf.predict_proba(va[feat_cols])[:,1]; p_te = clf.predict_proba(te[feat_cols])[:,1]
print(f"trees stopped at: {clf.best_iteration}")
print(f"val  PR-AUC: {average_precision_score(yva,p_va):.4f}   (logistic: 0.8291)")
print(f"test PR-AUC: {average_precision_score(yte,p_te):.4f}   (logistic: 0.4118)")
m = (va.is_auto_renew==1) & (va.is_free==0)
print(f"SAFE segment val: PR-AUC={average_precision_score(yva[m],p_va[m]):.4f}   (logistic: 0.0630)")
print(pd.Series(clf.feature_importances_, index=feat_cols).sort_values(ascending=False))

trees stopped at: 142
val  PR-AUC: 0.8422   (logistic: 0.8291)
test PR-AUC: 0.4528   (logistic: 0.4118)
SAFE segment val: PR-AUC=0.1097   (logistic: 0.0630)
is_free               0.669674
actual_amount_paid    0.184941
is_auto_renew         0.077289
discount              0.016516
recency_days          0.014711
activity_trend        0.007375
unq_30                0.006426
payment_plan_days     0.005363
active_days_30        0.005077
plan_list_price       0.004957
has_activity_60d      0.003397
secs_30               0.002820
completion_ratio      0.001455
dtype: float32


#### xgboost (same 13 feats)
- beats logistic everywhere; biggest on honest test 0.41→0.45 and safe segment 0.063→0.11 (interactions pay off where predicted)
- stopped at 142 trees = real structure
- gain importance: is_free 0.67 dominates, recency collapses to 0.015
  - NOT "engagement useless": gain rewards the one big root split (is_free cleaves 93%-churn trials); engagement's value lives in the small hard paid segment gain can't see
  - collinear pairs: tree picked amount_paid over list/plan_days, recency over has_activity (as predicted)
  - => global gain is the wrong lens; SHAP (phase C) for real segment attribution

In [18]:
# reuses clf / p_va / p_te / va / te / yva / yte from the cell above
for name, dfx, y, p in [('val', va, yva, p_va), ('test', te, yte, p_te)]:
    paid = (dfx.is_free == 0)                      # the realistic deployment population
    print(f"{name} paid-only: n={paid.sum()}, base={y[paid].mean():.4f}, "
          f"PR-AUC={average_precision_score(y[paid], p[paid]):.4f}   (topline {name}: "
          f"{average_precision_score(y, p):.4f})")

val paid-only: n=91305, base=0.0441, PR-AUC=0.2956   (topline val: 0.8422)
test paid-only: n=98499, base=0.0526, PR-AUC=0.2861   (topline test: 0.4528)


#### interrogation: is it just a free-trial detector? (largely yes)
- drop trials → val 0.84→0.296, test 0.45→0.286; ~2/3 of topline was the is_free root split (93%-churn trials)
- KEY: paid val≈test (0.296 vs 0.286) → topline val/test gap was trial-MIX, not model drift; model is stable on paid
- paid base ~5% (val 4.4 / test 5.3) vs topline ~10%; trials inflate base
- honest deployable number: ~0.29 PR-AUC on paid, ~6-7x no-skill, stable across time
- refines phase-A drift: much of it was composition (fewer trials in test), paid churn ~flat

DECISION (scope): model targets PAID customers only (is_free=0).
- free trials handled by RULE: is_free=1 → ~93% churn → flag for a conversion flow, not a retention offer
- rationale: cost model assumes a paying customer (~₹800 value); trials have no revenue to retain and need a different intervention; trials are trivially separable, so a rule beats burying them in the model AND de-inflates the headline metric
- train on paid-only so model capacity focuses on the hard problem
- (folds into DECISIONS.md at cleanup)

In [19]:
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
             'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
dfp = q("SELECT * FROM model_data WHERE is_free = 0")
tr, va, te = dfp[dfp.split=='train'], dfp[dfp.split=='val'], dfp[dfp.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
safe = (va.is_auto_renew == 1)
sc = StandardScaler().fit(tr[feat_cols])
lr = LogisticRegression(max_iter=2000).fit(sc.transform(tr[feat_cols]), ytr)
lr_va, lr_te = lr.predict_proba(sc.transform(va[feat_cols]))[:,1], lr.predict_proba(sc.transform(te[feat_cols]))[:,1]
xgb = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, importance_type='gain', n_jobs=-1, random_state=42)
xgb.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)
xg_va, xg_te = xgb.predict_proba(va[feat_cols])[:,1], xgb.predict_proba(te[feat_cols])[:,1]
print(f"paid base rate    : val {yva.mean():.4f}  test {yte.mean():.4f}")
print(f"logistic-paid     : val {average_precision_score(yva,lr_va):.4f}  test {average_precision_score(yte,lr_te):.4f}")
print(f"xgboost-paid      : val {average_precision_score(yva,xg_va):.4f}  test {average_precision_score(yte,xg_te):.4f}   (xgb-all-on-paid: 0.296/0.286)")
print(f"xgboost safe slice: val {average_precision_score(yva[safe],xg_va[safe]):.4f}  (n={safe.sum()})")
print(f"trees stopped at  : {xgb.best_iteration}")
print(pd.Series(xgb.feature_importances_, index=feat_cols).sort_values(ascending=False))

paid base rate    : val 0.0441  test 0.0526
logistic-paid     : val 0.2527  test 0.2379
xgboost-paid      : val 0.2964  test 0.2867   (xgb-all-on-paid: 0.296/0.286)
xgboost safe slice: val 0.1081  (n=80918)
trees stopped at  : 176
is_auto_renew         0.609323
actual_amount_paid    0.083315
recency_days          0.062008
payment_plan_days     0.050923
discount              0.049938
active_days_30        0.039822
activity_trend        0.033480
plan_list_price       0.027826
has_activity_60d      0.015216
unq_30                0.013407
secs_30               0.007908
completion_ratio      0.006834
dtype: float32


#### refit on paid-only
- xgboost-paid 0.296/0.287 == xgb-all-on-paid → focusing capacity gave NOTHING; is_free split was "free", not capacity-stealing
- => 0.29 is the FEATURE ceiling, not population/capacity; scoping was for honesty+economics, not accuracy
- val≈test stable; logistic floor 0.25, tree +0.05; recency climbed 0.015→0.062 once is_free gone
- is_auto_renew now tops gain (0.61): SAME pattern as is_free (biggest binary cleave, auto_renew=0 ~29% churn)
  - but DON'T scope it out — auto_renew=0 = paying + high-risk + retainable = the actual target (unlike trials)
- tiered honesty: all 0.84 → paid 0.30 → safe core 0.11
- lever to lift the hard core = better FEATURES (tenure/lifecycle/history), not tuning

In [22]:
con.execute("""
CREATE OR REPLACE TABLE feat_lifecycle AS
WITH pts AS (SELECT msno, expiry FROM cohorts_s),
     tx  AS (SELECT msno, strptime(transaction_date::VARCHAR,'%Y%m%d')::DATE AS txn_date FROM transactions)
SELECT p.msno, p.expiry,
       date_diff('day', MIN(t.txn_date), p.expiry) AS tenure_days,
       COUNT(DISTINCT t.txn_date)                   AS n_prior_cycles
FROM pts p JOIN tx t ON t.msno = p.msno AND t.txn_date <= p.expiry
GROUP BY p.msno, p.expiry
""")
print(q("SELECT COUNT(*) AS n FROM feat_lifecycle"))
print(q("SELECT MIN(tenure_days) lo, MAX(tenure_days) hi, MIN(n_prior_cycles) clo, MAX(n_prior_cycles) chi FROM feat_lifecycle"))
print(q("""
SELECT q AS tenure_quartile, ROUND(AVG(is_churn::INT),3) AS churn_rate, COUNT(*) AS n FROM (
  SELECT c.is_churn, NTILE(4) OVER (ORDER BY f.tenure_days) AS q
  FROM feat_lifecycle f JOIN cohorts_s c USING (msno, expiry)
) GROUP BY q ORDER BY q
"""))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

        n
0  499935
   lo   hi  clo  chi
0   0  789    1   57
   tenure_quartile  churn_rate       n
0                1       0.242  124984
1                2       0.052  124984
2                3       0.074  124984
3                4       0.043  124983


#### feat_lifecycle (tenure_days, n_prior_cycles; point-in-time)
- tenure 0–789, cycles 1–57, sane; 1:1 (−65 pts junk dates, immaterial)
- STRONG nonlinear signal: tenure Q1 24% vs Q2-4 ~5% = new-customer cliff, not fatigue
- caveat: inflated by trials in Q1; real test is on paid
- newer = flightier decisively; trees will love the cliff

In [23]:
num_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
            'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend',
            'tenure_days','n_prior_cycles']
cat_cols = ['payment_method_id']; feat_cols = num_cols + cat_cols
dfp = q("""SELECT m.*, f.tenure_days, f.n_prior_cycles FROM model_data m LEFT JOIN feat_lifecycle f USING (msno, expiry) WHERE m.is_free = 0""")
for c in cat_cols: dfp[c] = dfp[c].astype('category')
tr, va, te = dfp[dfp.split=='train'], dfp[dfp.split=='val'], dfp[dfp.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
safe = (va.is_auto_renew == 1)
xgb = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, importance_type='gain',
                    enable_categorical=True, tree_method='hist', n_jobs=-1, random_state=42)
xgb.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)
xg_va, xg_te = xgb.predict_proba(va[feat_cols])[:,1], xgb.predict_proba(te[feat_cols])[:,1]
print(f"paid base          : val {yva.mean():.4f}  test {yte.mean():.4f}")
print(f"xgboost +lifecycle : val {average_precision_score(yva,xg_va):.4f}  test {average_precision_score(yte,xg_te):.4f}   (prev paid: 0.296/0.287)")
print(f"safe slice (AR=1)  : val {average_precision_score(yva[safe],xg_va[safe]):.4f}   (prev: 0.108)")
print(f"trees stopped at   : {xgb.best_iteration}")
print(pd.Series(xgb.feature_importances_, index=feat_cols).sort_values(ascending=False))

paid base          : val 0.0441  test 0.0526
xgboost +lifecycle : val 0.2574  test 0.2620   (prev paid: 0.296/0.287)
safe slice (AR=1)  : val 0.0623   (prev: 0.108)
trees stopped at   : 1
n_prior_cycles        0.208133
payment_method_id     0.147670
is_auto_renew         0.106947
plan_list_price       0.087419
recency_days          0.077826
discount              0.072605
actual_amount_paid    0.071420
tenure_days           0.053321
active_days_30        0.051451
activity_trend        0.039984
unq_30                0.035214
payment_plan_days     0.026995
secs_30               0.010332
has_activity_60d      0.007137
completion_ratio      0.003548
dtype: float32


In [24]:
# 1) censoring fingerprint: is tenure bounded by the split (i.e. by calendar)?
print(dfp.groupby('split')['tenure_days'].agg(['mean','max']))

# 2) ablation: add ONE family at a time, watch best_iteration
base12 = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
          'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
def run(cols, cats=()):
    for c in cats: dfp[c] = dfp[c].astype('category')
    m = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                      eval_metric='aucpr', early_stopping_rounds=50, enable_categorical=bool(cats),
                      tree_method='hist', n_jobs=-1, random_state=42)
    m.fit(tr[cols], ytr, eval_set=[(va[cols], yva)], verbose=False)
    pv = m.predict_proba(va[cols])[:,1]
    return f"iters={m.best_iteration:>4}  val={average_precision_score(yva,pv):.4f}  safe={average_precision_score(yva[safe],pv[safe]):.4f}"

print("base12             :", run(base12))
print("base12 + lifecycle :", run(base12+['tenure_days','n_prior_cycles']))
print("base12 + paymethod :", run(base12+['payment_method_id'], cats=['payment_method_id']))

             mean  max
split                 
test   484.307323  789
train  241.358329  608
val    430.124091  699
base12             : iters= 131  val=0.2960  safe=0.1066
base12 + lifecycle : iters= 189  val=0.3103  safe=0.1207
base12 + paymethod : iters=  28  val=0.2896  safe=0.0904


#### ablation: what caused iters=1 (I guessed wrong)
- predicted lifecycle censoring; data says opposite
- censoring IS real: tenure max train 608 < val 699 < test 789 (bounded by 2015-01 start) → tenure partly a clock
- BUT lifecycle HELPS: val 0.296→0.310, safe 0.107→0.121, iters 131→189 — loyalty signal > calendar noise
- payment_method_id = poison: iters→28 alone, →1 combined, val/safe drop (high-card categorical, train-memorized, temporal mix shift)
- action: KEEP lifecycle (pending test confirm), DROP payment_method_id

In [25]:
safe_te = (te.is_auto_renew == 1)
def run_full(cols):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                      eval_metric='aucpr', early_stopping_rounds=50, tree_method='hist', n_jobs=-1, random_state=42)
    m.fit(tr[cols], ytr, eval_set=[(va[cols], yva)], verbose=False)
    pv, pt = m.predict_proba(va[cols])[:,1], m.predict_proba(te[cols])[:,1]
    return (f"iters={m.best_iteration:>4} | val={average_precision_score(yva,pv):.4f} "
            f"test={average_precision_score(yte,pt):.4f} | "
            f"safe_val={average_precision_score(yva[safe],pv[safe]):.4f} "
            f"safe_test={average_precision_score(yte[safe_te],pt[safe_te]):.4f}")

print("base12             :", run_full(base12))
print("base12 + lifecycle :", run_full(base12 + ['tenure_days','n_prior_cycles']))

base12             : iters= 131 | val=0.2960 test=0.2888 | safe_val=0.1066 safe_test=0.0780
base12 + lifecycle : iters= 189 | val=0.3103 test=0.2979 | safe_val=0.1207 safe_test=0.0943


#### lifecycle LOCKED (confirmed on test)
- +lifecycle beats base12 on ALL metrics incl honest test: test 0.289→0.298, safe_test 0.078→0.094 (~20% on hardest honest slice)
- censoring-clock worry didn't materialize; loyalty signal generalizes across the time gap
- WORKING MODEL = paid pop, 14 feats (12 base + tenure_days + n_prior_cycles), xgboost; payment_method_id DROPPED

In [26]:
# refit + STORE the locked working model (14 feats, paid population)
feat_cols = base12 + ['tenure_days', 'n_prior_cycles']
xgb = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, tree_method='hist', n_jobs=-1, random_state=42)
xgb.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)
p_va = xgb.predict_proba(va[feat_cols])[:, 1]      # for fitting the calibrator next step
p_te = xgb.predict_proba(te[feat_cols])[:, 1]      # for honest evaluation

# RAW reliability check on test: does predicted prob match observed churn, bin by bin?
chk = pd.DataFrame({'p': p_te, 'y': yte.values})
chk['bin'] = pd.qcut(chk['p'], 10, duplicates='drop')
rel = chk.groupby('bin', observed=True).agg(pred=('p', 'mean'), actual=('y', 'mean'), n=('y', 'size'))
print(rel.round(4))
print(f"\nmax raw predicted prob on test: {p_te.max():.4f}")   # foreshadows the 0.63 problem

                                    pred  actual     n
bin                                                   
(0.0016099999999999999, 0.00585]  0.0046  0.0133  9851
(0.00585, 0.00796]                0.0071  0.0233  9931
(0.00796, 0.0108]                 0.0095  0.0224  9768
(0.0108, 0.0129]                  0.0118  0.0235  9850
(0.0129, 0.0153]                  0.0139  0.0246  9850
(0.0153, 0.02]                    0.0175  0.0263  9849
(0.02, 0.031]                     0.0243  0.0330  9850
(0.031, 0.0539]                   0.0437  0.0488  9956
(0.0539, 0.121]                   0.0803  0.0721  9744
(0.121, 0.991]                    0.2776  0.2392  9850

max raw predicted prob on test: 0.9910


#### raw calibration (test) = S-curve
- low/mid bins UNDER-confident (pred 0.5–2.4% where actual 1.3–3.3%) — imbalance pushes scores to 0
- top bins OVER-confident (top bin pred 0.278 vs actual 0.239; raw max 0.99)
- isotonic will fix the S
- cost-rule implication: deflating the over-confident top means even fewer cross 0.63

In [27]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

iso = IsotonicRegression(out_of_bounds='clip').fit(p_va, yva)    # learn score→true-prob on val
p_te_cal = iso.predict(p_te)

# 1) reliability AFTER calibration — pred should now track actual
chk = pd.DataFrame({'p': p_te_cal, 'y': yte.values})
chk['bin'] = pd.qcut(chk['p'], 10, duplicates='drop')
print(chk.groupby('bin', observed=True).agg(pred=('p','mean'), actual=('y','mean'), n=('y','size')).round(4))

# 2) ranking preserved (isotonic is monotonic) + calibration improved
print(f"\nPR-AUC raw {average_precision_score(yte,p_te):.4f} -> cal {average_precision_score(yte,p_te_cal):.4f}  (should ~match)")
print(f"Brier   raw {brier_score_loss(yte,p_te):.5f} -> cal {brier_score_loss(yte,p_te_cal):.5f}  (lower=better)")

# 3) economics reality check: who clears the 0.625 break-even once probabilities are honest?
print(f"\ncalibrated max prob: {p_te_cal.max():.4f}")
for thr in [0.63, 0.30, 0.20, 0.10]:
    n = (p_te_cal >= thr).sum()
    print(f"  P>={thr:.2f}: {n:>5} customers ({n/len(p_te_cal)*100:5.2f}% of test)")

                     pred  actual      n
bin                                     
(-0.001, 0.00652]  0.0043  0.0158  14814
(0.00652, 0.0103]  0.0098  0.0258   5511
(0.0103, 0.0121]   0.0121  0.0246  22044
(0.0121, 0.0154]   0.0148  0.0225  13405
(0.0154, 0.0186]   0.0186  0.0281   8154
(0.0186, 0.0335]   0.0262  0.0367   5917
(0.0335, 0.0574]   0.0480  0.0524  14537
(0.0574, 0.0809]   0.0798  0.0897   4769
(0.0809, 1.0]      0.2411  0.2489   9348

PR-AUC raw 0.2979 -> cal 0.2859  (should ~match)
Brier   raw 0.04308 -> cal 0.04295  (lower=better)

calibrated max prob: 1.0000
  P>=0.63:   528 customers ( 0.54% of test)
  P>=0.30:  2410 customers ( 2.45% of test)
  P>=0.20:  3703 customers ( 3.76% of test)
  P>=0.10:  7238 customers ( 7.35% of test)


#### isotonic calibration (fit val, eval test)
- works WHERE IT MATTERS: top bins pred≈actual → decision region trustworthy
- low/mid still under on test → likely val→test DRIFT (val 4.4% < test 5.3%); Phase F recalibrate on recent data
- PR-AUC 0.298→0.286 (isotonic ties); keep 0.298 as model quality, calibrated probs for DECISION only
- economics: P≥0.63 → 528 cust (0.54%); rule works but tiny bc value≈1mo not LTV; list scales w/ value

In [28]:
p_va_cal = iso.predict(p_va)
print(f"VAL : mean cal prob {p_va_cal.mean():.4f} vs actual {yva.mean():.4f}  (calibrator's own set → should match)")
print(f"TEST: mean cal prob {p_te_cal.mean():.4f} vs actual {yte.mean():.4f}  (under-predicts if test churns more)")

VAL : mean cal prob 0.0441 vs actual 0.0441  (calibrator's own set → should match)
TEST: mean cal prob 0.0429 vs actual 0.0526  (under-predicts if test churns more)


#### drift confirmed 
- val mean cal prob = actual exactly; test under-predicts by ~0.01 = the base-rate gap
- => calibration transfers only as well as calib set represents deployment; Phase F = recalibrate on recent data

In [29]:
offer, save = 150, 0.30
paid_mask = te.payment_plan_days > 0
arpu = (te.loc[paid_mask, 'actual_amount_paid'] / te.loc[paid_mask, 'payment_plan_days'] * 30).median()
print(f"data-derived monthly ARPU (paid, median): ₹{arpu:.0f}\n")

rows = []
for horizon in [6, 12, 18, 24]:
    value = arpu * horizon
    be    = offer / (save * value)                       # break-even prob for this value
    mask  = p_te_cal >= be                                # the cost-optimal contact list
    n     = int(mask.sum())
    churn_in_list = int(yte.values[mask].sum())           # actual churners we'd have contacted
    precision    = churn_in_list / n if n else 0
    realized_net = churn_in_list * save * value - n * offer   # backtest on REAL outcomes
    rows.append({'horizon_mo': horizon, 'value_₹': round(value), 'break_even': round(be, 3),
                 'contacted': n, 'pct': round(n/len(p_te_cal)*100, 2),
                 'precision': round(precision, 3), 'realized_net_₹': round(realized_net)})
print(pd.DataFrame(rows).to_string(index=False))

data-derived monthly ARPU (paid, median): ₹149

 horizon_mo  value_₹  break_even  contacted  pct  precision  realized_net_₹
          6      894       0.559        770 0.78      0.675           23964
         12     1788       0.280       2828 2.87      0.463          277948
         18     2682       0.186       4208 4.27      0.390          689149
         24     3576       0.140       5753 5.84      0.336         1209700


#### decision layer (cost-optimal contact, backtested on test)
- ARPU ₹149/mo from data — matches standard monthly plan, grounded
- profitable at EVERY horizon: net ₹24k→₹1.2M as value rises → robust to the value assumption
- precision >> 5% base: 0.675 @ be0.56 (13x) → 0.336 @ be0.14 (6x); realized, assumption-free → model earns its keep
- tradeoff: ↑value → ↓break-even → bigger list, lower precision, higher total net (each add still +EV)
- @ defensible ₹149×12mo=₹1788: be 0.28, contact 2.9%, precision 46%, net ~₹278k (sample-scale)
- caveats: ₹ are sample-scale; save_rate 0.30 = softest assumption → Phase D uplift validates